# IEOR 4004 Current Optimization Model - Google Colab Runner

**Team 34 | Milestone 4 | Current tract-level model**

This notebook runs the current implementation directly from the team's GitHub repository. It intentionally does not copy the older standalone notebook logic, because that version predates the facility-specific municipal operating hours and the uniform 20% installation limit.

Current base case: Census-tract demand, 0.5-mile radius, equity weight \(\lambda=1\), 5% annual growth over five years, 60 kWh/EV/week, and 25% public-L2 share. The model first maximizes equity-weighted served demand and then minimizes investment cost while preserving the Stage 1 optimum.

## 1. Install Python packages

Run this cell once after opening the notebook in Colab.

In [ ]:
%pip install -q pandas numpy openpyxl geopy gurobipy

## 2. Load the latest project code

This cell clones the latest `main` branch. If the repository already exists in the Colab session, it performs a fast-forward pull instead.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/Hanruitang20/IEOR4004-project-Energy-Team34.git"
PROJECT_ROOT = Path("/content/IEOR4004-project-Energy-Team34")

if PROJECT_ROOT.exists():
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(PROJECT_ROOT)], check=True)

os.chdir(PROJECT_ROOT)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Project folder:", PROJECT_ROOT)
print("Git commit:", commit)

## 3. Configure a Gurobi license if required

The model has more variables than the size-limited Gurobi license permits. Use a valid academic, institutional, or Web License Service license. For WLS, add these three values to Colab **Secrets** and enable notebook access: `GRB_WLSACCESSID`, `GRB_WLSSECRET`, and `GRB_LICENSEID`.

This cell reads the values from Colab Secrets without printing or committing them. It creates a temporary `/tmp/gurobi.lic` file in the current runtime so Gurobi subprocesses can use WLS; the file disappears when the Colab runtime ends. If your Colab environment already has a working Gurobi license, you may skip it.

In [ ]:
import os
from pathlib import Path

secret_names = ["GRB_WLSACCESSID", "GRB_WLSSECRET", "GRB_LICENSEID"]
loaded = []
try:
    from google.colab import userdata
    for name in secret_names:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = str(value)
            loaded.append(name)
except ImportError:
    pass

if len(loaded) == 3:
    license_path = Path("/tmp/gurobi.lic")
    license_path.write_text(
        f"WLSACCESSID={os.environ['GRB_WLSACCESSID']}\n"
        f"WLSSECRET={os.environ['GRB_WLSSECRET']}\n"
        f"LICENSEID={os.environ['GRB_LICENSEID']}\n"
    )
    os.environ["GRB_LICENSE_FILE"] = str(license_path)
    print("Loaded Gurobi WLS credentials from Colab Secrets into a temporary runtime license file.")
elif loaded:
    print("Loaded only:", ", ".join(loaded))
else:
    print("No Colab WLS secrets loaded. Continue only if another valid Gurobi license is available.")

In [ ]:
import gurobipy as gp
print("Gurobi version:", ".".join(map(str, gp.gurobi.version())))
print("License check will be completed when the optimization model starts.")

## 4. Run the automated pipeline tests

The current repository should report nine passing tests.

In [ ]:
subprocess.run(["python", "-m", "unittest", "discover", "-s", "tests"], cwd=PROJECT_ROOT, check=True)

## 5. Run the current tract-level Gurobi model

The implementation is in `src/data_preparation.py` and `src/optimization.py`. This command uses `config/tract_case.json` and writes a fresh Colab run to `results/colab_tract_case/`.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "results" / "colab_tract_case"
proc = subprocess.run([
    "python", "run_model.py",
    "--config", "config/tract_case.json",
    "--output-dir", str(OUTPUT_DIR),
], cwd=PROJECT_ROOT, text=True, capture_output=True)
print(proc.stdout)
print(proc.stderr)
proc.check_returncode()

## 6. Review and verify the solution

In [ ]:
import json
import pandas as pd
from IPython.display import display

summary = json.loads((OUTPUT_DIR / "summary.json").read_text())
checks = json.loads((OUTPUT_DIR / "sanity_checks.json").read_text())

summary_fields = [
    "solver_status", "primary_solver_status", "primary_mip_gap", "final_mip_gap",
    "demand_points", "supply_sites", "accessible_pairs",
    "selected_municipal_sites", "additional_ports", "investment_cost_usd",
    "total_served_kwh", "projected_demand_kwh", "budget_slack_usd",
    "primary_stage1_objective",
]
display(pd.DataFrame({"value": {key: summary[key] for key in summary_fields}}))
display(pd.DataFrame({"value": checks}))

In [ ]:
# Compare the fresh Colab run with the current checked-in reference result.
expected = json.loads((PROJECT_ROOT / "results" / "tract_case" / "summary.json").read_text())
expected_checks = json.loads((PROJECT_ROOT / "results" / "tract_case" / "sanity_checks.json").read_text())

exact_fields = [
    "solver_status", "primary_solver_status", "demand_points", "supply_sites",
    "accessible_pairs", "selected_municipal_sites", "additional_ports",
    "investment_cost_usd",
]
for key in exact_fields:
    assert summary[key] == expected[key], f"Mismatch in {key}: {summary[key]} != {expected[key]}"

for key in ["total_served_kwh", "projected_demand_kwh", "primary_stage1_objective"]:
    assert abs(summary[key] - expected[key]) <= 1e-4, f"Mismatch in {key}"

assert checks["all_checks_pass"] is True
assert expected_checks["all_checks_pass"] is True
assert checks["maximum_flow_pair_distance_miles"] <= 0.5 + 1e-9
print("PASS: Fresh Colab results match the current checked-in tract reference result.")

## 7. Inspect the selected municipal facilities

In [ ]:
plan = pd.read_csv(OUTPUT_DIR / "municipal_candidate_solution.csv")
selected = plan.loc[plan["selected"].eq(1)].copy()
columns = [
    "facility_id", "site_name", "operating_hours_per_week", "existing_ports",
    "max_additional_ports", "additional_ports", "investment_cost_usd", "served_kwh",
]
display(selected[columns].sort_values(["additional_ports", "facility_id"], ascending=[False, True]).reset_index(drop=True))

## 8. Download the fresh output files

This creates a ZIP containing `summary.json`, `sanity_checks.json`, the selected-site plan, tract service results, and site utilization.

In [ ]:
import shutil

archive = shutil.make_archive("/content/IEOR4004_colab_tract_results", "zip", root_dir=OUTPUT_DIR)
print("Created:", archive)

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Automatic download is available when this notebook runs in Google Colab.")

## Implementation map

- `src/data_preparation.py`: loads the datasets, allocates ZIP registrations to tracts, applies facility-specific operating hours, constructs capacities and equity scores, and creates accessible pairs.
- `src/optimization.py`: creates the Gurobi decision variables, objective, budget, activation, projected-demand and capacity constraints; performs the two-stage solve; and writes sanity checks.
- `config/tract_case.json`: stores the current scenario assumptions, including the 20% municipal installation limit.
- `run_model.py`: command-line entry point used by this notebook.

Important limitations: tract internal points are spatial approximations; the current accessible-pair rule uses Haversine distance without separate physical-barrier screening; equity is a charging-supply-accessibility proxy; and cost/capacity inputs are planning assumptions rather than construction guarantees.